In [5]:
%pip install rank-bm25 nltk

from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize

corpus = [
    "AI detects phishing emails using text patterns",
    "Cybersecurity teams use machine learning to detect malware",
    "Banks use AI systems to identify credit card fraud",
    "Phishing email detection using AI and NLP",
    "Machine learning algorithms can classify malware files",
    "Fraud detection systems analyze unusual transaction behavior",
    "AI models can recognize phishing links in emails",
    "Malware detection with machine learning techniques",
    "Credit card fraud prediction using AI models"
]

tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

query = "AI for phishing email detection"
tokenized_query = word_tokenize(query.lower())
scores = bm25.get_scores(tokenized_query)

for doc, score in zip(corpus, scores):
    print(f"Document: {doc}\nScore: {score}\n")

query = "malware"
tokenized_query = word_tokenize(query.lower())
scores = bm25.get_scores(tokenized_query)

for doc, score in zip(corpus, scores):
    print(f"Document: {doc}\nScore: {score}\n")

Note: you may need to restart the kernel to use updated packages.
Document: AI detects phishing emails using text patterns
Score: 0.9800349235599779

Document: Cybersecurity teams use machine learning to detect malware
Score: 0.0

Document: Banks use AI systems to identify credit card fraud
Score: 0.309315052279613

Document: Phishing email detection using AI and NLP
Score: 3.3828231278975767

Document: Machine learning algorithms can classify malware files
Score: 0.0

Document: Fraud detection systems analyze unusual transaction behavior
Score: 0.6319657812035689

Document: AI models can recognize phishing links in emails
Score: 0.922259938983298

Document: Malware detection with machine learning techniques
Score: 0.6742011180661839

Document: Credit card fraud prediction using AI models
Score: 0.34806914235640896

Document: AI detects phishing emails using text patterns
Score: 0.0

Document: Cybersecurity teams use machine learning to detect malware
Score: 0.5947101565474634

Documen


[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pypdf import PdfReader

pdf_path = r"C:\Users\admin\Downloads\NLP lec\LEC\test.pdf"   # change only if your file name is different

reader = PdfReader(pdf_path)

documents = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text:
        documents.append({
            "doc_id": page_number + 1,
            "text": text
        })

print("Total pages loaded:", len(documents))
print(documents[0]["text"][:500])

Total pages loaded: 595
Praise for Hands-On Large Language Models
This is an exceptional guide to the world of language models and their
practical applications in industry. Its highly-visual coverage of
generative, representational, and retrieval applications of language
models empowers readers to quickly understand, use, and refine LLMs.
Highly recommended!
—Nils Reimers, Director of Machine Learning at Cohere |
creator of sentence-transformers
Jay and Maarten have continued their tradition of providing beautifully
il


In [8]:
import re
import math
from collections import Counter, defaultdict
import pandas as pd

In [9]:
class SimpleBM25ChunkSearch:
    def __init__(self, documents, chunk_size=5, overlap=1, k1=1.5, b=0.75):
        """
        documents: list of strings
        chunk_size: number of words per chunk
        overlap: repeated words between chunks
        """
        self.documents = documents
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.k1 = k1
        self.b = b

        self.chunks = []
        self.chunk_doc_ids = []

        self._create_chunks()
        self._build_bm25()

    def tokenize(self, text):
        text = text.lower()
        return re.findall(r"\b\w+\b", text)

    def _create_chunks(self):
        for doc_id, doc in enumerate(self.documents):
            tokens = self.tokenize(doc)

            start = 0
            while start < len(tokens):
                end = start + self.chunk_size
                chunk_tokens = tokens[start:end]

                if chunk_tokens:
                    self.chunks.append(chunk_tokens)
                    self.chunk_doc_ids.append(doc_id)

                start += self.chunk_size - self.overlap

    def _build_bm25(self):
        self.N = len(self.chunks)
        self.chunk_lengths = [len(chunk) for chunk in self.chunks]
        self.avgdl = sum(self.chunk_lengths) / self.N

        self.term_freqs = []
        self.doc_freq = defaultdict(int)

        for chunk in self.chunks:
            tf = Counter(chunk)
            self.term_freqs.append(tf)

            for term in tf:
                self.doc_freq[term] += 1

        self.idf = {}
        for term, df in self.doc_freq.items():
            self.idf[term] = math.log(1 + (self.N - df + 0.5) / (df + 0.5))

    def score_chunk(self, query_tokens, chunk_index):
        score = 0
        tf = self.term_freqs[chunk_index]
        dl = self.chunk_lengths[chunk_index]

        for term in query_tokens:
            if term not in tf:
                continue

            freq = tf[term]
            idf = self.idf.get(term, 0)

            numerator = freq * (self.k1 + 1)
            denominator = freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)

            score += idf * numerator / denominator

        return score

    def search(self, query):
        query_tokens = self.tokenize(query)

        chunk_results = []

        for i, chunk in enumerate(self.chunks):
            score = self.score_chunk(query_tokens, i)

            chunk_results.append({
                "doc_id": self.chunk_doc_ids[i],
                "chunk_id": i,
                "chunk_text": " ".join(chunk),
                "chunk_score": score
            })

        chunk_df = pd.DataFrame(chunk_results)

        doc_df = (
            chunk_df
            .groupby("doc_id", as_index=False)["chunk_score"]
            .sum()
            .rename(columns={"chunk_score": "document_score"})
            .sort_values("document_score", ascending=False)
        )

        doc_df["document_text"] = doc_df["doc_id"].apply(lambda i: self.documents[i])

        return chunk_df.sort_values("chunk_score", ascending=False), doc_df

In [11]:
%pip install pypdf

from pypdf import PdfReader

pdf_path = r"C:\Users\admin\Downloads\NLP lec\LEC\test.pdf"

pdfreader = PdfReader(pdf_path)
texts = []
for page in pdfreader.pages:
    text = page.extract_text()
    sentences = text.split(".")
    for sentence in sentences:
        sentence = sentence.strip()
        if sentence:
            texts.append((sentence, 0))  # Dummy label

documents = [sentence for sentence, label in texts]



bm25 = SimpleBM25ChunkSearch(
    documents=documents,
    chunk_size=3,
    overlap=1
)

query = "love ai"

chunk_scores, document_scores = bm25.search(query)


[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [ ]:
chunk_scores

document_scores

print("Query:", query)

print("\nChunk Scores:")
display(chunk_scores)

print("\nFinal Document Scores:")
display(document_scores)

Query: love ai

Chunk Scores:


,doc_id,chunk_id,chunk_text,chunk_score
9973,1130,9973,or love it,8.362944
21758,2626,21758,i love to,8.362944
25361,3046,25361,resilience love and,8.362944
21797,2631,21797,i love to,8.362944
9957,1130,9957,california love print_recommendations,8.362944
...,...,...,...,...
51504,5859,51504,is dalton maag,0.000000
51505,5859,51505,maag s ubuntu,0.000000
51506,5859,51506,ubuntu mono,0.000000
51507,5860,51507,oceanofpdf,0.000000



Final Document Scores:


,doc_id,document_score,document_text
5782,5790,67.654221,"L\nLangChain, Advanced Text Generation Techniq..."
5794,5802,36.360461,"natural language inference (NLI), Generating C..."
13,13,25.084758,"Well done!\n—Chris Fregly, Principal Solution ..."
5785,5793,22.551407,"supervised classification, Supervised Classifi..."
5819,5827,22.551407,"word-level metrics, in generative model evalua..."
...,...,...,...
5849,5857,0.000000,"The series design is by\nEdie Freedman, Ellie ..."
5850,5858,0.000000,The cover\nfonts are Gilroy Semibold and Guard...
5851,5859,0.000000,The text font is Adobe\nMinion Pro; the headin...
5852,5860,0.000000,OceanofPDF


In [ ]:
from pypdf import PdfReader

pdf_path = r"C:\Users\admin\Downloads\NLP lec\LEC\test.pdf"

reader = PdfReader(pdf_path)

pages = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text and text.strip():
        pages.append({
            "page_number": page_number + 1,
            "text": text
        })

print("Pages loaded:", len(pages))

Pages loaded: 595


In [ ]:
import re

def tokenize(text):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

def make_chunks(text, chunk_size=100, overlap=20):
    words = tokenize(text)
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = words[start:end]

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
chunk_data = []

for page in pages:
    chunks = make_chunks(
        page["text"],
        chunk_size=200,
        overlap=20
    )

    for chunk_id, chunk_tokens in enumerate(chunks):
        chunk_data.append({
            "page_number": page["page_number"],
            "chunk_id": chunk_id,
            "tokens": chunk_tokens,
            "text": " ".join(chunk_tokens)
        })

print("Total chunks:", len(chunk_data))

Total chunks: 830


In [ ]:
from rank_bm25 import BM25Okapi

corpus = [chunk["tokens"] for chunk in chunk_data]

bm25 = BM25Okapi(corpus)

In [ ]:
query = "transformer architecture"

query_tokens = tokenize(query)

scores = bm25.get_scores(query_tokens)

In [ ]:
import pandas as pd

chunk_results = []

for i, score in enumerate(scores):
    chunk_results.append({
        "page_number": chunk_data[i]["page_number"],
        "chunk_id": chunk_data[i]["chunk_id"],
        "chunk_score": score,
        "chunk_text": chunk_data[i]["text"]
    })

chunk_scores = pd.DataFrame(chunk_results)

chunk_scores = chunk_scores.sort_values(
    by="chunk_score",
    ascending=False
)

chunk_scores.head(10)

,page_number,chunk_id,chunk_score,chunk_text
257,182,0,9.083060,an interesting family of models that leverage ...
204,143,0,8.797968,figure 3 21 attention combines the relevant in...
55,42,0,8.498549,together these building blocks create the tran...
214,151,0,7.355843,figure 3 29 a transformer block from the origi...
50,37,0,7.204018,as a result during the generation of ik hou va...
60,45,1,7.115449,icon to indicate its generative capabilities g...
256,181,0,6.829406,give such a model iteratively improving your p...
221,156,0,6.666246,figure 3 33 rotary positional embeddings are a...
821,592,0,6.351397,optimizing attention optimizing attention from...
523,361,1,6.076104,field of computer vision the method they came ...


In [ ]:
document_scores = (
    chunk_scores
    .groupby("page_number", as_index=False)["chunk_score"]
    .sum()
    .rename(columns={"chunk_score": "document_score"})
    .sort_values(by="document_score", ascending=False)
)

document_scores.head(10)

,page_number,document_score
42,45,10.395548
357,361,9.633288
179,182,9.083060
140,143,8.797968
39,42,8.498549
375,379,8.430687
148,151,7.355843
34,37,7.204018
178,181,6.829406
154,157,6.771646


In [ ]:
top_chunks = chunk_scores.head(5)

for _, row in top_chunks.iterrows():
    print("Page:", row["page_number"])
    print("Chunk:", row["chunk_id"])
    print("Score:", round(row["chunk_score"], 3))
    print(row["chunk_text"][:700])
    print("-" * 80)

Page: 151
Chunk: 1
Score: 9.364
transformer architecture another improvement in
--------------------------------------------------------------------------------
Page: 182
Chunk: 0
Score: 9.298
an interesting family of models that leverage this architecture is the text to text transfer transformer or t5 model illustrated in figure 4 19 its architecture is similar to the original transformer where 12 decoders and 12 encoders are stacked together 7 figure 4 19 the t5 architecture is similar to the original transformer model a decoder encoder architecture with this architecture these models were first pretrained using masked language modeling in the first step of training illustrated in figure 4 20 instead of masking individual tokens sets of tokens or token spans were masked during pretraining
--------------------------------------------------------------------------------
Page: 42
Chunk: 0
Score: 9.045
together these building blocks create the transformer architecture and are the foundat

In [ ]:
%pip install sentence-transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

text = [
    "Ai detects algae blooms in satellite images",
    "Researchers use machine learning to predict protein folding",
    "Self-driving cars use AI to navigate complex traffic scenarios",
    "Ai sensors for algae bloom detection",
    "Machine learning algorithms for protein folding prediction",
    "Autonomous vehicles and AI navigation systems",
    "Algae bloom detection using AI and satellite imagery",
    "Predicting protein folding with machine learning techniques",
    "AI applications in self-driving car technology"
]

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [chunk["text"] for chunk in chunk_data]
chunk_embeddings = model.encode(chunk_texts)
query_embedding = model.encode([query])
similarity_scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()
chunk_results = []
for i, score in enumerate(similarity_scores):
    chunk_results.append({
        "page_number": chunk_data[i]["page_number"],
        "chunk_id": chunk_data[i]["chunk_id"],
        "similarity_score": score,
        "chunk_text": chunk_data[i]["text"]
    })

chunk_scores = pd.DataFrame(chunk_results)
chunk_scores = chunk_scores.sort_values(
    by="similarity_score",
    ascending=False
)
chunk_scores.head(10)

query_embedding = model.encode([query])
similarity_scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()
document_results = []
for i, score in enumerate(similarity_scores):
    document_results.append({
        "page_number": chunk_data[i]["page_number"],
        "similarity_score": score,
        "chunk_text": chunk_data[i]["text"]
    })

document_scores = pd.DataFrame(document_results)
document_scores = (
    document_scores
    .groupby("page_number", as_index=False)["similarity_score"]
    .sum()
    .rename(columns={"similarity_score": "document_score"})
    .sort_values(by="document_score", ascending=False)
)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17211.69it/s]


In [ ]:
document_scores = pd.DataFrame(document_results)
document_scores = (
    document_scores
    .groupby("page_number", as_index=False)["similarity_score"]
    .sum()
    .rename(columns={"similarity_score": "document_score"})
    .sort_values(by="document_score", ascending=False)
)
document_scores.head(10)

,page_number,document_score
154,157,0.852658
357,361,0.750268
140,143,0.718646
495,499,0.681543
118,121,0.672827
0,2,0.605630
148,151,0.586980
155,158,0.567097
35,38,0.525879
379,383,0.516037


In [ ]:
from pypdf import PdfReader

pdf_path = r"C:\Users\admin\Downloads\NLP lec\LEC\test.pdf"

reader = PdfReader(pdf_path)

pages = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text and text.strip():
        pages.append({
            "page_number": page_number + 1,
            "text": text
        })

print("Pages loaded:", len(pages))

Pages loaded: 595
